# Flujo máximo

En una red con capacidades en las aristas, el flujo máximo de una fuente $s$ a un sumidero $t$ es la mayor cantidad que puede circular respetando las capacidades y conservándose en cada nodo intermedio. Ford-Fulkerson repite: buscar un camino aumentante en el grafo residual y empujar por él su capacidad mínima. Con BFS (Edmonds-Karp) cuesta $O(V \cdot E^2)$. Por el teorema de flujo máximo y corte mínimo, el valor del flujo máximo es igual a la capacidad del corte $(S, T)$ más barato que separa $s$ de $t$.

## 1. Edmonds-Karp

Implementar Edmonds-Karp y calcular cuántos litros por segundo llegan del depósito al barrio en una red de tuberías.

In [1]:
import random
from collections import defaultdict, deque

def edmonds_karp(capacidad, s, t):
    residual = defaultdict(lambda: defaultdict(int))
    for u in capacidad:
        for v, c in capacidad[u].items():
            residual[u][v] += c
            residual[v][u] += 0
    flujo = 0
    while True:
        padre, cola = {s: None}, deque([s])
        while cola and t not in padre:
            u = cola.popleft()
            for v, c in residual[u].items():
                if c > 0 and v not in padre:
                    padre[v] = u
                    cola.append(v)
        if t not in padre:
            return flujo, residual
        camino, v = [], t
        while padre[v] is not None:
            camino.append((padre[v], v))
            v = padre[v]
        empuje = min(residual[u][v] for u, v in camino)
        for u, v in camino:
            residual[u][v] -= empuje
            residual[v][u] += empuje
        flujo += empuje

tuberias = {"depósito": {"A": 10, "B": 5}, "A": {"B": 15, "C": 9}, "B": {"D": 10}, "C": {"barrio": 10}, "D": {"C": 6, "barrio": 10}}
flujo, residual = edmonds_karp(tuberias, "depósito", "barrio")
print("flujo máximo:", flujo, "l/s")

flujo máximo: 15 l/s


## 2. Corte mínimo

A partir del grafo residual final, los nodos alcanzables desde la fuente forman $S$. Listar las aristas del corte y comprobar que su capacidad total coincide con el flujo máximo.

In [2]:
def corte_minimo(capacidad, residual, s):
    S, pila = {s}, [s]
    while pila:
        u = pila.pop()
        for v, c in residual[u].items():
            if c > 0 and v not in S:
                S.add(v)
                pila.append(v)
    return S, [(u, v) for u in capacidad for v in capacidad[u] if u in S and v not in S]

S, corte = corte_minimo(tuberias, residual, "depósito")
print("S =", sorted(S))
print("corte:", corte)
assert sum(tuberias[u][v] for u, v in corte) == flujo

S = ['depósito']
corte: [('depósito', 'A'), ('depósito', 'B')]


## 3. Comprobar con fuerza bruta

En redes aleatorias pequeñas, el flujo máximo debe ser igual al mínimo de las capacidades de todos los cortes posibles (todos los subconjuntos $S$ con $s \in S$, $t \notin S$).

In [3]:
from itertools import combinations

def corte_bruto(capacidad, nodos, s, t):
    intermedios = [u for u in nodos if u not in (s, t)]
    mejor = float("inf")
    for r in range(len(intermedios) + 1):
        for extra in combinations(intermedios, r):
            S = {s, *extra}
            mejor = min(mejor, sum(c for u in capacidad for v, c in capacidad[u].items() if u in S and v not in S))
    return mejor

random.seed(1)
for _ in range(300):
    n = random.randint(2, 7)
    cap = {u: {v: random.randint(1, 9) for v in range(n) if v != u and random.random() < 0.4} for u in range(n)}
    f, _ = edmonds_karp(cap, 0, n - 1)
    assert f == corte_bruto(cap, range(n), 0, n - 1)
print("ok")

ok


## 4. Emparejamiento como flujo

Resolver la asignación de voluntarios a turnos como un problema de flujo: una fuente unida a cada voluntario, cada voluntario a sus turnos y cada turno al sumidero, todas con capacidad 1. Recuperar qué voluntario cubre cada turno.

In [4]:
disponibilidad = {"Ana": ["lunes", "martes"], "Luis": ["lunes"], "Eva": ["martes", "jueves"], "Pau": ["lunes"], "Noa": ["jueves", "viernes"]}
red = {"fuente": {v: 1 for v in disponibilidad}}
for voluntario, turnos in disponibilidad.items():
    red[voluntario] = {turno: 1 for turno in turnos}
for turno in {t for ts in disponibilidad.values() for t in ts}:
    red[turno] = {"sumidero": 1}

f, residual = edmonds_karp(red, "fuente", "sumidero")
asignacion = {v: t for v in disponibilidad for t in disponibilidad[v] if residual[v][t] == 0}
print(f, asignacion)
assert f == len(asignacion) == 4

4 {'Ana': 'martes', 'Luis': 'lunes', 'Eva': 'jueves', 'Noa': 'viernes'}
